<a href="https://colab.research.google.com/github/WahabBasa/Detect-Retina-Damage-Using-Transfer-Learning/blob/main/resnet50-transfer-learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retinal OCT damage detection with ResNet50

Optical coherence tomography (OCT) takes cross-sectional images of the retina. Around 30 million scans
are performed each year and reading them is slow specialist work, which makes automated triage worth
testing. This notebook uses an ImageNet-pretrained **ResNet50** as a frozen feature extractor with a small
trainable classification head, sorting scans into four classes:

| Class | Meaning |
|---|---|
| `CNV` | Choroidal neovascularization — abnormal new blood vessels under the retina (wet age-related macular degeneration) |
| `DME` | Diabetic macular edema — fluid build-up and retinal thickening caused by diabetes |
| `DRUSEN` | Yellow deposits under the retina, the early marker of age-related macular degeneration |
| `NORMAL` | Healthy retina |

The data is the Kermany 2018 OCT set (`paultimothymooney/kermany2018`): 83,484 training, 968 test and
32 validation images.

**What this experiment tests:** how far a frozen ImageNet backbone plus a shallow head gets on OCT
imagery, and how ResNet50 at 224x224 inputs compares against the same recipe built on
InceptionV3. The counterpart notebook is [`inceptionv3-transfer-learning.ipynb`](https://github.com/WahabBasa/Detect-Retina-Damage-Using-Transfer-Learning/blob/main/inceptionv3-transfer-learning.ipynb).

## Setup

In [ ]:
import os
import random

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# The Kaggle archive really does unpack into a directory whose name ends in a space.
DATA_ROOT = '/content/OCT2017 '
CLASS_NAMES = ['NORMAL', 'CNV', 'DME', 'DRUSEN']
IMG_SIZE = (224, 224)
IMAGES_PER_CLASS = 2000
BATCH_SIZE = 124
EPOCHS = 80
CHECKPOINT_PATH = 'best_resnet50_model.keras'

print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## Data

The download needs a Kaggle API token (Kaggle → Settings → API → Create New Token). The upload cell is
Colab-specific; running locally, skip it and put `kaggle.json` at `~/.kaggle/kaggle.json` with
permissions `600`. The archive is about 10.8 GB.

Three choices in the loader are deliberate:

- **Images are loaded as raw 0–255 arrays** and each model then applies its *own* `preprocess_input`.
  ResNet50's Keras weights are caffe-mode: `resnet50.preprocess_input` flips RGB to BGR and subtracts the ImageNet channel means from 0–255 values. Feeding it `/255` RGB instead — as an earlier version of this notebook did — leaves every filter looking at the wrong input distribution and pins training accuracy near chance.
- **Class balance comes from capped equal sampling** — 2,000 images per class — so the training
  set is balanced by construction. No oversampling or class weighting is applied, because with equal
  caps both would be no-ops.
- **File lists are shuffled with a fixed seed before truncation.** `os.listdir` order is arbitrary and
  Kermany filenames encode patient IDs, so taking the first N would cluster the sample on a handful of
  patients.

The `val` split drives the callbacks and the `test` split is left untouched until the single final
evaluation at the bottom.

In [ ]:
import shutil

# Colab: this opens a file picker — choose the kaggle.json you downloaded from Kaggle.
# Running locally: skip this cell entirely (see the note above).
from google.colab import files

files.upload()

KAGGLE_CONFIG_DIR = '/root/.config/kaggle'
os.makedirs(KAGGLE_CONFIG_DIR, exist_ok=True)
shutil.copy('kaggle.json', os.path.join(KAGGLE_CONFIG_DIR, 'kaggle.json'))
os.chmod(os.path.join(KAGGLE_CONFIG_DIR, 'kaggle.json'), 0o600)

In [ ]:
import kaggle

kaggle.api.dataset_download_files('paultimothymooney/kermany2018', path='.', unzip=True)
print('Dataset downloaded.')

In [ ]:
for split in ['train', 'val', 'test']:
    counts = {c: len(os.listdir(os.path.join(DATA_ROOT, split, c))) for c in CLASS_NAMES}
    print(f'{split:<5} {sum(counts.values()):>6}  {counts}')

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.utils import to_categorical


def load_split(split, max_images_per_class=None, seed=SEED):
    """Load one split as raw 0-255 float arrays; the caller applies model-specific preprocessing."""
    images, labels = [], []
    rng = random.Random(seed)
    for class_index, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATA_ROOT, split, class_name)
        file_names = sorted(os.listdir(class_dir))
        rng.shuffle(file_names)
        if max_images_per_class:
            file_names = file_names[:max_images_per_class]
        for img_name in file_names:
            img = load_img(os.path.join(class_dir, img_name), target_size=IMG_SIZE)
            images.append(img_to_array(img))
            labels.append(class_index)
    return np.array(images), np.array(labels)


x_train, y_train_int = load_split('train', max_images_per_class=IMAGES_PER_CLASS)
x_val, y_val_int = load_split('val')
x_test, y_test_int = load_split('test')

# ResNet50's own preprocessing, applied in place so we do not hold two copies of the arrays.
x_train = preprocess_input(x_train)
x_val = preprocess_input(x_val)
x_test = preprocess_input(x_test)

y_train = to_categorical(y_train_int, num_classes=len(CLASS_NAMES))
y_val = to_categorical(y_val_int, num_classes=len(CLASS_NAMES))
y_test = to_categorical(y_test_int, num_classes=len(CLASS_NAMES))

for split_name, y in [('train', y_train_int), ('val', y_val_int), ('test', y_test_int)]:
    counts = {c: int(np.sum(y == i)) for i, c in enumerate(CLASS_NAMES)}
    print(f'{split_name:<5} {len(y):>6}  {counts}')

## Model

The ResNet50 backbone is frozen — only the two dense layers and the softmax output train — so this is
feature extraction rather than full fine-tuning.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(len(CLASS_NAMES), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

## Training

`ModelCheckpoint` keeps the best-validation weights on disk and `EarlyStopping` halts a run that stops
improving. One caveat worth naming: the official `val` split is only 32 images (8 per class), so
validation metrics are coarse and checkpoint selection is noisier than it looks. It is used as-is here
so the 968-image `test` split stays genuinely held out.

`model.fit` shuffles the training data every epoch, so the class-ordered arrays above are fine as loaded.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint = ModelCheckpoint(CHECKPOINT_PATH, save_best_only=True, monitor='val_accuracy', mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint, early_stop],
)

## Evaluation

The saved checkpoint is reloaded before scoring. `EarlyStopping(restore_best_weights=True)` only
restores anything when it actually triggers a stop, so a run that finishes all its epochs would
otherwise be evaluated on its final — often overfit — weights.

This is the one and only pass over the full `test` split.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import load_model

model = load_model(CHECKPOINT_PATH)

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Test loss {loss:.4f} | test accuracy {accuracy:.4f}\n')

y_pred = np.argmax(model.predict(x_test), axis=1)

print(classification_report(y_test_int, y_pred, target_names=CLASS_NAMES, digits=3))
print('Confusion matrix (rows = true, columns = predicted)')
print(confusion_matrix(y_test_int, y_pred))